In [ ]:
"""
Train Gravitational Wave Detection Model

Simple training script that loads data and trains the CNN.
Run this after generating the dataset.

Usage:
    python src/train.py
"""

import numpy as np
import os
from tensorflow import keras
from tensorflow.keras import layers, models

# Paths
BASE_DIR = os.path.dirname(os.path.dirname(__file__))
DATASET_DIR = os.path.join(BASE_DIR, 'data', 'dataset_qtransform')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

def load_data():
    """Load preprocessed training data"""
    X_train = np.load(os.path.join(DATASET_DIR, 'X_train.npy'))
    X_test = np.load(os.path.join(DATASET_DIR, 'X_test.npy'))
    y_train = np.load(os.path.join(DATASET_DIR, 'y_train.npy'))
    y_test = np.load(os.path.join(DATASET_DIR, 'y_test.npy'))
    return X_train, X_test, y_train, y_test

def build_model(input_shape=(224, 224, 3)):
    """Build CNN architecture"""
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Conv blocks
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Dense layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

if __name__ == "__main__":
    print("Loading data...")
    X_train, X_test, y_train, y_test = load_data()
    print(f"Train: {len(X_train)}, Test: {len(X_test)}")
    
    print("\nBuilding model...")
    model = build_model()
    model.summary()
    
    print("\nTraining...")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=50,
        batch_size=16,
        verbose=1
    )
    
    print("\nSaving model...")
    model.save(os.path.join(MODELS_DIR, 'gw_detector.keras'))
    print("✅ Done!")